# Homework 4
-   **Name:**  Victor Hugo Gomez Soto 
-  **e-mail:** -- victor.gomez2701@alumnos.udg.mx --


# MODULES

In [3]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from scipy.spatial import distance
from scipy.stats import wrapcauchy, levy_stable
import math
import dash
from dash import dcc, html
from dash.dependencies import Input, Output



In [4]:
# Nota: Esta clase la importaremos junto con el segundo bloque de modulos
################# http://www.pygame.org/wiki/2DVectorClass ##################
class Vec2d(object):
    """2d vector class, supports vector and scalar operators,
       and also provides a bunch of high level functions
       """
    __slots__ = ['x', 'y']

    def __init__(self, x_or_pair, y = None):
        if y == None:            
            self.x = x_or_pair[0]
            self.y = x_or_pair[1]
        else:
            self.x = x_or_pair
            self.y = y
            
    # Addition
    def __add__(self, other):
        if isinstance(other, Vec2d):
            return Vec2d(self.x + other.x, self.y + other.y)
        elif hasattr(other, "__getitem__"):
            return Vec2d(self.x + other[0], self.y + other[1])
        else:
            return Vec2d(self.x + other, self.y + other)

    # Subtraction
    def __sub__(self, other):
        if isinstance(other, Vec2d):
            return Vec2d(self.x - other.x, self.y - other.y)
        elif (hasattr(other, "__getitem__")):
            return Vec2d(self.x - other[0], self.y - other[1])
        else:
            return Vec2d(self.x - other, self.y - other)
    
    # Vector length
    def get_length(self):
        return math.sqrt(self.x**2 + self.y**2)
    
    # rotate vector
    def rotated(self, angle):        
        cos = math.cos(angle)
        sin = math.sin(angle)
        x = self.x*cos - self.y*sin
        y = self.x*sin + self.y*cos
        return Vec2d(x, y)
     # Método para convertir el vector en una tupla
    def to_tuple(self):
        return (self.x, self.y)

In [ ]:
#####################################################################################
# Brownian motion trajectoy
#####################################################################################
def bm_2d(n_steps=1000, speed=5, start_pos=(0, 0)):    
    pos = Vec2d(*start_pos)
    trajectory = [pos.to_tuple()]
    
    for _ in range(n_steps):
        turn_angle = np.random.uniform(low=-np.pi, high=np.pi)
        step = Vec2d(speed, 0).rotated(turn_angle)
        pos += step
        trajectory.append(pos.to_tuple())
    
    df = pd.DataFrame(trajectory, columns=["x_pos", "y_pos"])
    return df
#####################################################################################
# Correlated Random Walk 
#####################################################################################
def rw_2d(n_steps=1000, speed=5, start_pos=(0, 0), c=0.5):   
    pos = Vec2d(*start_pos)
    trajectory = [pos.to_tuple()]
    
    angle = 0
    for _ in range(n_steps):
        delta_angle = wrapcauchy.rvs(c)
        angle += delta_angle
        step = Vec2d(speed, 0).rotated(angle)
        pos += step
        trajectory.append(pos.to_tuple())
    
    df = pd.DataFrame(trajectory, columns=["x_pos", "y_pos"])
    return df

#####################################################################################
# Levi Flight  
#####################################################################################
def levy_flight(n_steps=1000, alpha=1.5, scale=1.0, c=0.5, start_pos=(0, 0)):
    pos = Vec2d(*start_pos)
    trajectory = [pos.to_tuple()]
    angle = 0  

    for _ in range(n_steps):
        step_size = np.abs(levy_stable.rvs(alpha, 0, scale=scale))
        delta_angle = wrapcauchy.rvs(c)  
        angle += delta_angle
        step = Vec2d(step_size, 0).rotated(angle)
        pos += step
        trajectory.append(pos.to_tuple())

    df = pd.DataFrame(trajectory, columns=["x_pos", "y_pos"])
    return df

#####################################################################################
# path length 
#####################################################################################
def path_length(df):
    """Calcula la longitud total del camino recorrido."""
    distances = np.sqrt(np.diff(df["x_pos"])**2 + np.diff(df["y_pos"])**2)
    return np.sum(distances)


# Simulación de Trayectorias Aleatorias en 3D

## Introducción
Las trayectorias aleatorias se utilizan en múltiples campos científicos y de ingeniería para modelar movimientos impredecibles.
A continuación, exploraremos tres modelos fundamentales de trayectorias aleatorias: **Random Walk**, **Brownian Motion** y **Lévy Flight**.

---

## 1. Random Walk (Caminata Aleatoria)
### Definición
Un **Random Walk** es un modelo en el que un objeto se mueve paso a paso en una dirección aleatoria. 
Cada paso tiene una dirección y magnitud aleatorias, pero sin correlación con los pasos anteriores.

### Funcionamiento
- En cada paso, el objeto selecciona una nueva dirección aleatoria.
- Puede moverse en 1D, 2D o 3D.
- La distancia recorrida crece proporcionalmente a la raíz cuadrada del número de pasos: **d ≈ sqrt(N)**.

### Aplicaciones
- Modelado del movimiento de moléculas en gases y líquidos (difusión).
- Predicción de fluctuaciones en mercados financieros.
- Modelado de la dispersión de animales en ecología.

---

## 2. Brownian Motion (Movimiento Browniano)
### Definición
Es un tipo de **Random Walk** continuo en el que la posición de una partícula cambia de forma aleatoria en pequeños intervalos de tiempo.
Fue descrito matemáticamente por Albert Einstein en 1905.

### Funcionamiento
- Se genera mediante la suma de muchos pequeños movimientos aleatorios.
- Sigue un proceso estocástico con media cero y varianza creciente con el tiempo.
- Se modela con una ecuación diferencial estocástica (SDE).

### Diferencias con el Random Walk
- Es un proceso **continuo**, mientras que el Random Walk es discreto.
- Se utiliza para modelar sistemas donde los cambios pequeños son acumulativos.

### Aplicaciones
- Movimientos de partículas suspendidas en líquidos (Ejemplo: polen en agua).
- Predicción de precios de activos en finanzas (Modelo de Black-Scholes).
- Modelado de señales biológicas y tráfico en redes.

---

## 3. Lévy Flight (Vuelo de Lévy)
### Definición
Un **Lévy Flight** es una variación del **Random Walk** en la que los pasos siguen una distribución de **Lévy**.
Se caracteriza por **saltos largos ocasionales** en lugar de movimientos siempre pequeños.

### Funcionamiento
- En su mayoría, los pasos son pequeños, pero ocasionalmente ocurren grandes saltos.
- La distancia recorrida en un número dado de pasos es mayor que en un Brownian Motion.
- Sigue una distribución de Lévy con exponentes entre 1 y 3.

### Diferencias con el Brownian Motion
- **Brownian Motion** tiene pasos de tamaño similar, mientras que **Lévy Flight** introduce grandes saltos ocasionales.
- **Lévy Flight** es más eficiente para buscar en grandes áreas porque puede explorar más rápido.

### Aplicaciones
- Modelado del comportamiento de animales en búsqueda de alimento.
- Estrategias de búsqueda óptimas en inteligencia artificial y algoritmos de optimización.
- Predicción de fluctuaciones extremas en mercados financieros.

---

## Conclusión
- **Random Walk**: Pasos aleatorios sin memoria, buena aproximación de la difusión molecular.
- **Brownian Motion**: Modelo continuo de difusión con pequeños cambios acumulativos.
- **Lévy Flight**: Modelo de búsqueda eficiente con grandes saltos ocasionales.

Estos modelos son fundamentales en múltiples disciplinas, desde la física hasta la inteligencia artificial.


# Simulation of Random Trajectories in 3D

## Introduction
Random trajectories are used in multiple scientific and engineering fields to model unpredictable movements.  
Below, we explore three fundamental models of random trajectories: **Random Walk**, **Brownian Motion**, and **Lévy Flight**.

---

## 1. Random Walk
### Definition
A **Random Walk** is a model where an object moves step by step in a random direction.  
Each step has a random direction and magnitude, with no correlation to previous steps.

### How It Works
- At each step, the object selects a new random direction.
- It can move in 1D, 2D, or 3D.
- The distance traveled grows proportionally to the square root of the number of steps: **d ≈ sqrt(N)**.

### Applications
- Modeling molecular movement in gases and liquids (diffusion).
- Predicting fluctuations in financial markets.
- Modeling animal dispersion in ecology.

---

## 2. Brownian Motion
### Definition
**Brownian Motion** is a type of continuous **Random Walk** where a particle’s position changes randomly in small time intervals.  
It was mathematically described by Albert Einstein in 1905.

### How It Works
- It is generated by summing many small random movements.
- It follows a stochastic process with zero mean and increasing variance over time.
- It is modeled using a stochastic differential equation (SDE).

### Differences from Random Walk
- It is a **continuous** process, while Random Walk is discrete.
- It is used to model systems where small cumulative changes are significant.

### Applications
- Motion of particles suspended in liquids (e.g., pollen in water).
- Predicting asset prices in finance (Black-Scholes model).
- Modeling biological signals and network traffic.

---

## 3. Lévy Flight
### Definition
A **Lévy Flight** is a variation of **Random Walk** where steps follow a **Lévy distribution**.  
It is characterized by **occasional long jumps** instead of always taking small steps.

### How It Works
- Most steps are small, but occasionally, there are large jumps.
- The distance traveled in a given number of steps is greater than in Brownian Motion.
- It follows a Lévy distribution with exponents between 1 and 3.

### Differences from Brownian Motion
- **Brownian Motion** has steps of similar size, while **Lévy Flight** introduces occasional large jumps.
- **Lévy Flight** is more efficient for searching large areas as it explores faster.

### Applications
- Modeling animal foraging behavior.
- Optimal search strategies in artificial intelligence and optimization algorithms.
- Predicting extreme fluctuations in financial markets.

---

## Conclusion
- **Random Walk**: Random steps with no memory, a good approximation of molecular diffusion.
- **Brownian Motion**: Continuous diffusion model with small cumulative changes.
- **Lévy Flight**: Efficient search model with occasional large jumps.

These models are fundamental across multiple disciplines, from physics to artificial intelligence.


In [43]:
import dash
from dash import dcc, html
from dash.dependencies import Input, Output
import plotly.graph_objects as go
import pandas as pd
import numpy as np

class Vec3d:
    """3D vector class for trajectory simulation"""
    __slots__ = ['x', 'y', 'z']

    def __init__(self, x, y, z):
        self.x, self.y, self.z = x, y, z

    def __add__(self, other):
        return Vec3d(self.x + other.x, self.y + other.y, self.z + other.z)

    def rotated(self, angle_xy, angle_xz):
        """Rotar en los planos XY y XZ"""
        cos_xy, sin_xy = math.cos(angle_xy), math.sin(angle_xy)
        cos_xz, sin_xz = math.cos(angle_xz), math.sin(angle_xz)

        # Rotación en el plano XY
        x_new = self.x * cos_xy - self.y * sin_xy
        y_new = self.x * sin_xy + self.y * cos_xy

        # Rotación en el plano XZ
        z_new = self.z * cos_xz - self.x * sin_xz
        x_new = self.x * cos_xz + self.z * sin_xz  # Corregir x después de la rotación en XZ

        return Vec3d(x_new, y_new, z_new)

    def to_tuple(self):
        return (self.x, self.y, self.z)
#####################################################################################
# Brownian motion trajectoy
#####################################################################################
def bm_3d(n_steps=1000, speed=5, start_pos=(0, 0, 0)):
    pos = Vec3d(*start_pos)
    trajectory = [pos.to_tuple()]

    for _ in range(n_steps):
        turn_angle_xy = np.random.uniform(-np.pi, np.pi)
        turn_angle_xz = np.random.uniform(-np.pi, np.pi)
        step = Vec3d(speed, 0, 0).rotated(turn_angle_xy, turn_angle_xz)
        pos += step
        trajectory.append(pos.to_tuple())

    df = pd.DataFrame(trajectory, columns=["x_pos", "y_pos", "z_pos"])
    return df
#####################################################################################
# Correlated Random Walk 
#####################################################################################
def rw_3d(n_steps=1000, speed=5, start_pos=(0, 0, 0), c=0.5):   
    pos = Vec3d(*start_pos)
    trajectory = [pos.to_tuple()]

    angle_xy = 0  # Ángulo en el plano XY
    angle_xz = 0  # Ángulo en el plano XZ

    for _ in range(n_steps):
        delta_angle_xy = np.random.vonmises(mu=0, kappa=c)  # Variación del ángulo XY
        delta_angle_xz = np.random.vonmises(mu=0, kappa=c)  # Variación del ángulo XZ

        angle_xy += delta_angle_xy
        angle_xz += delta_angle_xz

        step = Vec3d(speed, 0, 0).rotated(angle_xy, angle_xz)  # Movimiento en 3D
        pos += step
        trajectory.append(pos.to_tuple())

    df = pd.DataFrame(trajectory, columns=["x_pos", "y_pos", "z_pos"])
    return df

#####################################################################################
# Levi Flight  
#####################################################################################
def levy_flight_3d(n_steps=1000, alpha=1.5, scale=1.0, c=0.5, start_pos=(0, 0, 0)):
    pos = Vec3d(*start_pos)  # Iniciar en 3D
    trajectory = [pos.to_tuple()]
    
    angle_xy = 0  # Ángulo en el plano XY
    angle_xz = 0  # Ángulo en el plano XZ

    for _ in range(n_steps):
        step_size = np.abs(levy_stable.rvs(alpha, 0, scale=scale))  # Paso de Lévy
        delta_angle_xy = np.random.vonmises(mu=0, kappa=c)  # Variación del ángulo en XY
        delta_angle_xz = np.random.vonmises(mu=0, kappa=c)  # Variación del ángulo en XZ

        angle_xy += delta_angle_xy
        angle_xz += delta_angle_xz

        step = Vec3d(step_size, 0, 0).rotated(angle_xy, angle_xz)  # Movimiento en 3D
        pos += step
        trajectory.append(pos.to_tuple())

    df = pd.DataFrame(trajectory, columns=["x_pos", "y_pos", "z_pos"])
    
    return df

#####################################################################################
# path length 
#####################################################################################
def path_length_3d(trajectory):    
    # Obtener la distancia euclidiana entre cada par de puntos consecutivos en 3D
    distances = np.array([
        distance.euclidean(trajectory.iloc[i - 1], trajectory.iloc[i])
        for i in range(1, trajectory.shape[0])
    ])

    # Devolver la suma acumulativa de las distancias
    return np.cumsum(distances)
#####################################################################################
# turning angle distribution
#####################################################################################

def turning_angle_distribution_3d(df):
    
    # Diferencias de posición en X, Y y Z
    dx = df["x_pos"].diff().values[1:]  # Omitimos el primer valor NaN
    dy = df["y_pos"].diff().values[1:]
    dz = df["z_pos"].diff().values[1:]

    # Construcción de los vectores de movimiento en 3D
    v1 = np.column_stack((dx[:-1], dy[:-1], dz[:-1]))  # Primeros desplazamientos
    v2 = np.column_stack((dx[1:], dy[1:], dz[1:]))  # Segundos desplazamientos

    # Normalizamos los vectores
    norm_v1 = np.linalg.norm(v1, axis=1)
    norm_v2 = np.linalg.norm(v2, axis=1)

    # Evitar división por cero
    valid_indices = (norm_v1 > 0) & (norm_v2 > 0)
    v1, v2 = v1[valid_indices], v2[valid_indices]
    norm_v1, norm_v2 = norm_v1[valid_indices], norm_v2[valid_indices]

    # Producto punto y ángulos en 3D
    dot_product = np.einsum("ij,ij->i", v1, v2)
    cos_theta = dot_product / (norm_v1 * norm_v2)  # Cálculo del coseno del ángulo
    cos_theta = np.clip(cos_theta, -1, 1)  # Evitar errores numéricos fuera de [-1,1]

    angles = np.arccos(cos_theta)  # Convertir a ángulos en radianes
    return np.degrees(angles)  # Convertir a grados

#####################################################################################
# mean squared displacement
#####################################################################################
def mean_squared_displacement_3d(df):

    x = df['x_pos'].values
    y = df['y_pos'].values
    z = df['z_pos'].values  # Se agrega la tercera dimensión
    N = len(x)
    msd = np.zeros(N)

    for t in range(N):
        dx = x[t:] - x[:N-t]  # Desplazamiento en X
        dy = y[t:] - y[:N-t]  # Desplazamiento en Y
        dz = z[t:] - z[:N-t]  # Desplazamiento en Z
        squared_displacement = dx**2 + dy**2 + dz**2  # MSD = dx² + dy² + dz²
        msd[t] = np.mean(squared_displacement)

    return msd

# Inicializar la app Dash
app = dash.Dash(__name__)

# Definir función para generar una trayectoria
def generate_trajectory(traj_type='BM', n_steps=500, speed=5, alpha=1.5, scale=1.0, c=0.5):
    if traj_type == 'BM':
        df = bm_3d(n_steps, speed)
    elif traj_type == 'CRW':
        df = rw_3d(n_steps, speed, c=c)
    else:
        df = levy_flight_3d(n_steps, alpha, scale, c)
    return df

# Estilos para mejorar la UI
styles = {
    'container': {
        'width': '80%',
        'margin': 'auto',
        'padding': '20px',
        'fontFamily': 'Arial, sans-serif',
        'backgroundColor': '#f8f9fa',
        'borderRadius': '10px',
        'boxShadow': '0px 0px 10px rgba(0, 0, 0, 0.1)'
    },
    'header': {
        'textAlign': 'center',
        'fontSize': '24px',
        'fontWeight': 'bold',
        'marginBottom': '20px'
    },
    'panel': {
        'padding': '15px',
        'border': '1px solid #ccc',
        'borderRadius': '5px',
        'backgroundColor': '#fff',
        'marginBottom': '15px'
    }
}

# Layout de la app
app.layout = html.Div(style=styles['container'], children=[
    html.H1("Simulación de Trayectorias Aleatorias", style=styles['header']),
    
    html.Div(style=styles['panel'], children=[
        html.Label("Selecciona el tipo de trayectoria:"),
        dcc.RadioItems(
            id='traj-selector',
            options=[
                {'label': 'Movimiento Browniano (BM)', 'value': 'BM'},
                {'label': 'Camino Aleatorio Correlacionado (CRW)', 'value': 'CRW'},
                {'label': 'Vuelo de Lévy (LF)', 'value': 'LF'}
            ],
            value='BM',
            inline=True
        )
    ]),

    html.Div(style=styles['panel'], children=[
        html.Label("Número de pasos:"),
        dcc.Slider(id='n-steps', min=100, max=1000, step=100, value=500, 
                   marks={i: str(i) for i in range(100, 1100, 200)})
    ]),

    html.Div(style=styles['panel'], children=[
        html.Label("Velocidad:"),
        dcc.Slider(id='speed', min=1, max=10, step=1, value=5)
    ]),

    html.Div(id='extra-params', style=styles['panel']),

    html.Div(style=styles['panel'], children=[
        html.Label("Selecciona la métrica a visualizar:"),
        dcc.Dropdown(
            id='metric-selector',
            options=[
                {'label': 'Path Length (PL)', 'value': 'PL'},
                {'label': 'Mean Squared Displacement (MSD)', 'value': 'MSD'},
                {'label': 'Turning Angle Distribution (TAD)', 'value': 'TAD'}
            ],
            value='PL'
        )
    ]),

    dcc.Graph(id='trajectory-plot'),
    dcc.Graph(id='metric-plot')
])

# Callback para mostrar parámetros adicionales
@app.callback(
    Output('extra-params', 'children'),
    [Input('traj-selector', 'value')]
)
def update_params(traj_type):
    if traj_type == 'BM':
        return ''
    return html.Div([
        html.Label("Coeficiente de Cauchy (CRW & LF):"),
        dcc.Slider(id='c-coefficient', min=0.1, max=1, step=0.1, value=0.5),
        html.Label("Exponente de Lévy (LF):"),
        dcc.Slider(id='alpha', min=0.5, max=2, step=0.1, value=1.5),
        html.Label("Escala (LF):"),
        dcc.Slider(id='scale', min=0.1, max=5, step=0.1, value=1.0)
    ])

# Callback para actualizar gráficos
@app.callback(
    [Output('trajectory-plot', 'figure'),
     Output('metric-plot', 'figure')],
    [Input('traj-selector', 'value'),
     Input('n-steps', 'value'),
     Input('speed', 'value'),
     Input('metric-selector', 'value')]
)
def update_plots(traj_type, n_steps, speed, metric):
    df = generate_trajectory(traj_type, n_steps, speed)  # Ahora usa la versión 3D

    # print(f"Path Length for {traj_type}: {path_length(df)}")

    # 🔷 **Gráfica de trayectoria en 3D**
    fig1 = go.Figure()
    fig1.add_trace(go.Scatter3d(
        x=df['x_pos'], 
        y=df['y_pos'], 
        z=df['z_pos'],  # Ahora agregamos la tercera dimensión
        mode='lines', 
        name='Trayectoria',
        line=dict(width=3)  # Línea más visible
    ))

    fig1.update_layout(
        title='Trayectoria en 3D',
        scene=dict(
            xaxis_title='X',
            yaxis_title='Y',
            zaxis_title='Z'
        )
    )

    # 🔷 **Gráfica de métrica seleccionada**
    if metric == 'PL':
        metric_value = path_length(df)
        print(f"Path Length for {traj_type}: {path_length(df)}")
        
        fig2 = go.Figure()
        fig2.add_trace(go.Scatter(
            x=np.arange(len(metric_value)),  # El índice como eje X (tiempo)
            y=metric_value,  
            mode='lines',
            name='Path Length'
        ))
        
        fig2.update_layout(
            title='Path Length',
            xaxis_title='Tiempo',
            yaxis_title='Distancia Acumulada',
            bargap=0.5  # Espaciado entre barras (no aplica aquí, pero mantiene el formato)
        )
    
    elif metric == 'MSD':
        metric_values = mean_squared_displacement_3d(df)
        
        fig2 = go.Figure()
        fig2.add_trace(go.Scatter(y=metric_values, mode='lines', name='MSD'))
        fig2.update_layout(title='Mean Squared Displacement')

    else:
        metric_values = turning_angle_distribution_3d(df)
        hist, bin_edges = np.histogram(metric_values, bins=30, density=True)
        bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])  

        fig2 = go.Figure()
        fig2.add_trace(go.Scatter(
            x=bin_centers, 
            y=hist, 
            mode='lines', 
            name="Densidad de Ángulos", 
            line=dict(color='blue')
        ))

        fig2.update_layout(
            title='Turning Angle Distribution',
            xaxis_title="Ángulo (grados)",
            yaxis_title="Densidad"
        )

    return fig1, fig2


# Ejecutar la aplicación
if __name__ == '__main__':
    app.run_server(debug=True)


Path Length for BM: [   6.42293733   12.11203267   18.70757172   25.45157776   31.91555031
   38.98173779   45.74093075   52.49066117   59.3587734    66.39002749
   71.43004625   78.4046411    85.39791445   91.92272052   98.78534718
  103.81375861  109.91848363  116.30740254  123.2095777   130.06134596
  135.22650055  141.59629948  146.81331172  153.3908719   160.29607258
  165.85733976  172.22669309  177.76073336  183.01583974  190.01074455
  196.43449184  202.03644298  207.09633649  212.28461419  218.55126288
  225.57792351  232.02618686  238.76281123  245.52993758  252.54895285
  259.41168826  264.4987043   269.56689948  275.61762094  282.251013
  287.27210848  294.16060663  301.14809009  306.15362788  312.85215121
  318.63313465  324.88468999  329.94325968  336.71453413  342.02535407
  348.32169087  354.01675974  359.3432881   366.32752611  372.54762848
  379.61705897  386.65011922  392.72230972  397.99269435  403.01021155
  410.06333298  416.90871147  421.91305966  428.38509875  4